In [ ]:
import sys

!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install librosa
!{sys.executable} -m pip install ruptures
!{sys.executable} -m pip install matplotlib

# Imports
import librosa as lb
import ruptures as rpt
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Constants
song_file_path = "/Users/shivamenta/Desktop/training_data copy/KETTAMA - It Gets Better - Forever Mix.mp3"
FREQUENCY_BUCKETS = [(0, 200), (201, 600), (601, 3000), (3001, 7000), (7001, 22000)]

STFT Change Point Detection

In [ ]:
y, sr = lb.load(song_file_path, sr=None)
stft_result = lb.stft(y, n_fft=2048, hop_length=512)
magnitude = np.abs(stft_result)
freqs = lb.fft_frequencies(sr=sr)

freq_bucket_to_signal = [[] for _ in range(len(FREQUENCY_BUCKETS))]
num_time_slices = stft_result.shape[1]

for t in range(num_time_slices):
    for i, (lower, upper) in enumerate(FREQUENCY_BUCKETS):
        freq_indices = np.where((freqs >= lower) & (freqs <= upper))[0]
        bucket_intensity = np.sum(magnitude[freq_indices, t])
        freq_bucket_to_signal[i].append(bucket_intensity)


# Smooth Out Signal
def moving_average(x, w):
    return np.convolve(x, np.ones(w), "valid") / w


for idx, bucket in enumerate(freq_bucket_to_signal):
    np_arr = np.array(bucket)
    freq_bucket_to_signal[idx] = moving_average(np_arr, 200)

In [ ]:
def plot_frequency_buckets(predicted_change_points=[], actual_change_points=[]):
    plt.figure(figsize=(10, 6))

    # Plot each frequency bucket's intensity over time
    for i, bucket in enumerate(freq_bucket_to_signal):
        plt.plot(
            bucket,
            label=f"Bucket {i+1}: {FREQUENCY_BUCKETS[i][0]}-{FREQUENCY_BUCKETS[i][1]} Hz",
        )

    # Label the axes and add a title
    plt.xlabel("Time (Frames)")
    plt.ylabel("Intensity")
    plt.title("Frequency Bucket Intensities Over Time")

    for cp in predicted_change_points:
        plt.axvline(
            x=cp,
            color="red",
            linestyle="--",
            label="Predicted Change Point" if cp == predicted_change_points[0] else "",
        )
    for cp in actual_change_points:
        plt.axvline(
            x=cp,
            color="blue",
            linestyle="--",
            label="Actual Change Point" if cp == predicted_change_points[0] else "",
        )

    # Add a legend to distinguish the different frequency buckets
    plt.legend()

    # Display the plot
    plt.tight_layout()
    plt.show()


# Plot the results
plot_frequency_buckets()

In [ ]:
# 200 100000000
mat = np.array(freq_bucket_to_signal).T
model = rpt.KernelCPD(kernel="linear", min_size=157).fit(mat)
estimated_change_points = model.predict(pen=100000000)

plot_frequency_buckets(estimated_change_points, [])
print(estimated_change_points)

Vocal Track Separation Demo (Spleeter)

In [ ]:
import sys

!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install librosa
!{sys.executable} -m pip install ruptures
!{sys.executable} -m pip install matplotlib

# Imports
import librosa as lb
import ruptures as rpt
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import librosa
import torch
import torchaudio
import matplotlib.pyplot as plt
from pathlib import Path
from demucs.pretrained import get_model
from demucs.apply import apply_model

song_file_mp3 = "/Users/shivamenta/Desktop/training_data/Super Bass.mp3"
import numpy as np
import librosa
import torch
import torchaudio
import matplotlib.pyplot as plt
from pathlib import Path
from demucs.pretrained import get_model
from demucs.apply import apply_model

def separate_vocals(audio_path):
    """
    Separate vocals from audio using Demucs Python API.
    
    Args:
        audio_path: Path to audio file
    
    Returns:
        vocals: Tensor of separated vocal track
        sr: Sample rate
    """
    print("Loading Demucs model...")
    # Load pretrained model (htdemucs is the default, best quality)
    model = get_model('htdemucs')
    model.eval()
    
    # Load audio file
    print(f"Loading audio from: {audio_path}")
    waveform, sr = torchaudio.load(audio_path)
    
    # Demucs expects stereo input, convert mono to stereo if needed
    if waveform.shape[0] == 1:
        waveform = waveform.repeat(2, 1)
    
    # Add batch dimension
    waveform = waveform.unsqueeze(0)
    
    print("Separating vocals (this may take 1-3 minutes)...")
    # Apply model
    with torch.no_grad():
        sources = apply_model(model, waveform, device='cpu', shifts=1, split=True)
    
    # Extract vocals (index 3 for htdemucs: drums=0, bass=1, other=2, vocals=3)
    vocals = sources[0, 3]  # Shape: [channels, samples]
    
    print("✓ Vocal separation complete!")
    return vocals, sr


def detect_vocal_activity(mp3_path, threshold_db=-40, frame_length=2048, hop_length=512):
    """
    Detect when vocals are active in an MP3 file using Demucs Python API.
    
    Args:
        mp3_path: Path to the MP3 file
        threshold_db: Energy threshold in dB (lower = more sensitive)
        frame_length: Frame size for analysis
        hop_length: Hop size between frames
    
    Returns:
        Dictionary with timestamps and vocal activity boolean array
    """
    
    # Step 1: Separate vocals using Demucs Python API
    vocals_tensor, sr = separate_vocals(mp3_path)
    
    # Step 2: Convert to numpy and mono for analysis
    print("Analyzing vocal track...")
    # Average stereo channels to mono
    vocals_numpy = vocals_tensor.mean(dim=0).numpy()
    
    # Resample to 22050 Hz for consistency with librosa
    vocals_numpy = librosa.resample(vocals_numpy, orig_sr=sr, target_sr=22050)
    sr = 22050
    
    # Step 3: Calculate frame-by-frame energy (RMS)
    rms = librosa.feature.rms(y=vocals_numpy, frame_length=frame_length, hop_length=hop_length)[0]
    
    # Convert to dB
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    
    # Step 4: Threshold to detect activity
    is_vocal_active = rms_db > threshold_db
    
    # Step 5: Convert frame indices to timestamps
    times = librosa.frames_to_time(np.arange(len(rms_db)), sr=sr, hop_length=hop_length)
    
    # Step 6: Create segments of continuous vocal activity
    segments = []
    in_segment = False
    start_time = 0
    
    for i, active in enumerate(is_vocal_active):
        if active and not in_segment:
            start_time = times[i]
            in_segment = True
        elif not active and in_segment:
            segments.append((start_time, times[i]))
            in_segment = False
    
    # Close last segment if needed
    if in_segment:
        segments.append((start_time, times[-1]))
    
    return {
        'times': times,
        'is_active': is_vocal_active,
        'rms_db': rms_db,
        'segments': segments,
        'sample_rate': sr,
        'vocals_audio': vocals_numpy
    }


def plot_vocal_activity(result):
    """Visualize the vocal activity detection with waveform."""
    fig, axes = plt.subplots(3, 1, figsize=(14, 9))
    
    # Create time array for waveform
    waveform_times = np.linspace(0, len(result['vocals_audio']) / result['sample_rate'], 
                                  len(result['vocals_audio']))
    
    # Plot 1: Waveform
    axes[0].plot(waveform_times, result['vocals_audio'], linewidth=0.5, alpha=0.7)
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('Vocal Track Waveform')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(0, waveform_times[-1])
    
    # Plot 2: RMS energy
    axes[1].plot(result['times'], result['rms_db'])
    axes[1].set_ylabel('Energy (dB)')
    axes[1].set_title('Vocal Track Energy (RMS)')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(0, result['times'][-1])
    
    # Plot 3: Vocal activity with shaded regions
    axes[2].fill_between(result['times'], 0, result['is_active'].astype(int), 
                          alpha=0.3, label='Vocal Active')
    axes[2].plot(result['times'], result['is_active'].astype(int), linewidth=2)
    axes[2].set_ylabel('Vocal Active')
    axes[2].set_xlabel('Time (seconds)')
    axes[2].set_title('Vocal Activity Detection')
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim(0, result['times'][-1])
    
    plt.tight_layout()
    plt.savefig('vocal_activity.png', dpi=150, bbox_inches='tight')
    print("Visualization saved as 'vocal_activity.png'")


def export_segments_to_file(segments, output_file='vocal_segments.txt'):
    """Export vocal segments to a text file."""
    with open(output_file, 'w') as f:
        f.write("Vocal Activity Segments\n")
        f.write("=" * 40 + "\n\n")
        for i, (start, end) in enumerate(segments, 1):
            duration = end - start
            f.write(f"Segment {i}: {start:6.2f}s - {end:6.2f}s (duration: {duration:.2f}s)\n")
    print(f"Segments exported to: {output_file}")


def save_vocal_track(vocals_audio, sr, output_path='separated_vocals.wav'):
    """Save the separated vocal track to a file."""
    import soundfile as sf
    sf.write(output_path, vocals_audio, sr)
    print(f"Vocal track saved to: {output_path}")


# Example usage
if __name__ == "__main__":
    # Analyze an MP3 file
    result = detect_vocal_activity(song_file_mp3, threshold_db=-20)
    
    # Print vocal segments
    print("\n" + "="*50)
    print("Vocal Activity Segments:")
    print("="*50)
    for i, (start, end) in enumerate(result['segments'], 1):
        duration = end - start
        print(f"Segment {i:2d}: {start:6.2f}s - {end:6.2f}s (duration: {duration:5.2f}s)")
    
    # Calculate statistics
    total_time = result['times'][-1]
    vocal_time = np.sum(result['is_active']) * (result['times'][1] - result['times'][0])
    vocal_percentage = (vocal_time / total_time) * 100
    
    print("\n" + "="*50)
    print("Statistics:")
    print("="*50)
    print(f"Total song duration: {total_time:.2f}s")
    print(f"Vocal active time: {vocal_time:.2f}s ({vocal_percentage:.1f}%)")
    print(f"Number of vocal segments: {len(result['segments'])}")
    
    # Save separated vocal track
    save_vocal_track(result['vocals_audio'], result['sample_rate'])
    
    # Export segments to file
    export_segments_to_file(result['segments'])
    
    # Create visualization
    plot_vocal_activity(result)
    
    print("\n✓ Analysis complete!")

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from typing import List
import os
import hashlib
from scipy.ndimage import gaussian_filter
import bisect
from functools import cache

# Rekordbox imports
from pyrekordbox import Rekordbox6Database
from track_interface import TrackInterface

SAMPLE_RATE = 1000
HOP_LENGTH = 50

def load_rekordbox_training_cuepoints(track_filepath: str) -> List[int]:
    db = Rekordbox6Database()
    playlist = db.get_playlist(Name="training_data").one()

    tgt_ti = None
    for song in playlist.Songs:
        ti = TrackInterface(song, db)
        if ti.get_content_filepath() == track_filepath:
            tgt_ti = ti
            break
    
    if not tgt_ti:
        raise Exception(f"No labeled data found for {track_filepath}")
    
    return [cue / 1000.0 for cue in tgt_ti.read_hot_cues()]

def load_rekordbox_first_beat_timestamps(track_filepath: str) -> List[int]:
    db = Rekordbox6Database()
    playlist = db.get_playlist(Name="training_data").one()

    tgt_ti = None
    for song in playlist.Songs:
        ti = TrackInterface(song, db)
        if ti.get_content_filepath() == track_filepath:
            tgt_ti = ti
            break
    
    if not tgt_ti:
        raise Exception(f"No labeled data found for {track_filepath}")
    
    return [cue / 1000.0 for cue in tgt_ti._get_first_beat_timestamps()]


def get_recurrence_matrix(track_filepath: str):
    # Compute MFCC and recurrence matrix
    print(f"Computing MFCC and recurrence matrix for {os.path.basename(track_filepath)}")
    
    # Load audio file
    y, sr = librosa.load(track_filepath, sr=SAMPLE_RATE)
    
    # Extract MFCC features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=HOP_LENGTH)
    
    # Compute recurrence matrix using MFCC features
    recurrence_matrix = librosa.segment.recurrence_matrix(mfcc,
                                                        metric='cosine',
                                                        mode='affinity',
                                                        sym=True)
    print(f"Recurrence matrix has size {len(recurrence_matrix)}")

    return recurrence_matrix

def calculate_optimal_kernel_size(sample_rate, hop_length, bpm):
    kernel_length = 8
    measure_length_in_seconds = 240 / bpm
    # essentially return how many hops in 1 measure time 8
    samples_per_second_in_feature_matrix = sample_rate / hop_length
    samples_per_measure = samples_per_second_in_feature_matrix / measure_length_in_seconds

    return samples_per_measure * kernel_length

def novelty_curve_changepoints(recurrence_matrix):
    def checkerboard_kernel(kernel_size: int):
        M = kernel_size
        kernel = np.outer(np.hanning(M), np.hanning(M))
        kernel[:M//2, :M//2] *= 1
        kernel[M//2:, M//2:] *= 1
        kernel[:M//2, M//2:] *= -1
        kernel[M//2:, :M//2] *= -1
        return kernel
    
    def novelty_times_from_matrix(R_smooth, reverse=False):
        pad_size = kernel_size // 2
        R_padded = np.pad(R_smooth, pad_size, mode='constant', constant_values=0)
        
        original_size = R_smooth.shape[0]
        
        if reverse:
            novelty = np.array([
                np.sum(R_padded[i:i+kernel_size, i:i+kernel_size] * kernel)
                for i in range(original_size - 1, -1, -1)
            ])
        else:
            novelty = np.array([
                np.sum(R_padded[i:i+kernel_size, i:i+kernel_size] * kernel)
                for i in range(original_size)
            ])

        # Remove negative values
        novelty = np.maximum(novelty, 0)
        
        # Apply moving average BEFORE normalization
        def moving_average(signal, window_size):
            return np.convolve(signal, np.ones(window_size)/window_size, mode='same')
        
        novelty = moving_average(novelty, 50)
        
        # Normalize after smoothing
        novelty = novelty / (novelty.max() + 1e-10)
        
        # Optional: Apply median filter to remove spurious spikes
        from scipy.ndimage import median_filter
        novelty = median_filter(novelty, size=15)

        novelty_times = librosa.frames_to_time(
            np.arange(len(novelty)), 
            sr=SAMPLE_RATE, 
            hop_length=HOP_LENGTH
        )

        # More restrictive peak detection
        prominence_threshold = np.percentile(novelty, 75) * 0.5

        peaks, properties = find_peaks(
            novelty, 
            distance=50,
            prominence=prominence_threshold,
            width=5,
            height=0.3
        )

        boundary_frames_novelty = peaks

        # Convert frames to time
        boundary_times_novelty = librosa.frames_to_time(
            boundary_frames_novelty, 
            sr=SAMPLE_RATE, 
            hop_length=HOP_LENGTH
        )

        return boundary_times_novelty, boundary_frames_novelty, novelty_times, novelty
    
    matrix_size = recurrence_matrix.shape[0]
    song_length = librosa.frames_to_time(
        matrix_size, 
        sr=SAMPLE_RATE, 
        hop_length=HOP_LENGTH
    )
    num_frames = matrix_size

    # Apply stronger smoothing to recurrence matrix
    R_smooth = gaussian_filter(recurrence_matrix, sigma=1.0)
    
    # Larger kernel for better structural detection
    kernel_size = 450
    kernel = checkerboard_kernel(kernel_size)
        
    boundary_times_novelty, boundary_frames_novelty, novelty_times, novelty = novelty_times_from_matrix(R_smooth)
    boundary_times_from_end_novelty, boundary_frames_from_end_novelty, _, _ = novelty_times_from_matrix(R_smooth, reverse=True)

    flipped_boundary_times, flipped_boundary_frames = [], []
    
    print(boundary_times_novelty, boundary_times_from_end_novelty)
    print(boundary_frames_novelty, boundary_frames_from_end_novelty)

    # Convert reverse frame indices to forward frame indices
    for i in range(len(boundary_frames_from_end_novelty)):
        # Reverse array index to forward frame index
        forward_frame = (num_frames - 1) - boundary_frames_from_end_novelty[i]
        flipped_boundary_frames.append(forward_frame)
        flipped_boundary_times.append(librosa.frames_to_time(
            forward_frame, 
            sr=SAMPLE_RATE, 
            hop_length=HOP_LENGTH
        ))
    
    # Plot novelty curve with changepoints
    plt.figure(figsize=(14, 5))
    plt.plot(novelty_times, novelty, linewidth=1.5, color='green', label='Novelty Curve')
    
    # Plot detected boundaries
    for time in boundary_times_novelty:
        plt.axvline(x=time, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    for time in flipped_boundary_times:
        plt.axvline(x=time, color='blue', linestyle='--', linewidth=1.5, alpha=0.7)
    
    # Mark peaks on the curve
    plt.scatter(boundary_times_novelty, novelty[boundary_frames_novelty], 
                color='red', s=50, zorder=5, label='Detected Boundaries (Forward)')

    plt.scatter(flipped_boundary_times, novelty[flipped_boundary_frames], 
                color='blue', s=50, zorder=5, label='Detected Boundaries (Reverse)')
    
    plt.title(f"Novelty Curve with Detected Changepoints ({len(boundary_frames_novelty)} forward, {len(flipped_boundary_frames)} reverse)")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Novelty")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return boundary_times_novelty

@cache
def get_smoothed_rms(filepath, smooth_window = 50):
    y, sr = librosa.load(filepath, sr=SAMPLE_RATE)
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)[0]
    
    # Smooth RMS using moving average
    if smooth_window > 1:
        kernel = np.ones(smooth_window) / smooth_window
        rms_smoothed = np.convolve(rms, kernel, mode="same")
        assert len(rms_smoothed) == len(rms), f"Length mismatch: {len(rms_smoothed)} vs {len(rms)}"
    else:
        rms_smoothed = rms
    
    return rms_smoothed

def get_rms_trend_of_section(rms, start, end):
    pass

def merge_sections_on_energy(filepath, recurrence_matrix, changepoints):
    # purpose - ensure we merge any portions of the track that are pretty similar
    end = 0
    smoothed_rms = get_smoothed_rms(filepath)
    new_changepoints = []

    prev_rms_trend = []
    # merge adjacent sections if they have the same RMS trends
    for idx in range(len(changepoints)):
        curr_start = changepoints[idx]
        if idx == len(changepoints):
            curr_end = end
        else:
            curr_end = changepoints[idx + 1]
        new_rms = get_rms_trend_of_section(smoothed_rms, curr_start, curr_end)
        if new_rms == prev_rms_trend:
            continue

        prev_rms_trend = new_rms
        new_changepoints.append(curr_start)
    
    return new_changepoints

def fit_changepoints_to_beatgrid(changepoints, first_beats):
    # need to figure out how to account for this difference
    return [first_beats[bisect.bisect_left(first_beats, changepoint)] for changepoint in changepoints]

def get_changepoints_from_recurrence_matrix(recurrence_matrix, cuepoints, first_beat_timestamps, track_path) -> List[int]:
    # boundary_times_novelty = novelty_curve_changepoints(recurrence_matrix)
    boundary_times_novelty = novelty_curve_changepoints(recurrence_matrix)

    # fit to first beat changepoints (can probably do some sort of assumption on 4s / 2s based on a majority algo for some that are close)
    
    return boundary_times_novelty


def plot_recurrence_matrix(recurrence_matrix: str, track_name: str, cuepoints: List[int], changepoints: List[int], filepath: str, smooth=False):
    # Define three different sigma values for Gaussian filters
    SIGMA = 1.0
    print("start visualizing")
    # Create figure with 2 subplots
    fig = plt.figure(figsize=(12, 10))
    

    # Apply Gaussian filter with current sigma value
    if smooth:
        recurrence_matrix_filtered = gaussian_filter(recurrence_matrix, sigma=SIGMA)
    else:
        recurrence_matrix_filtered = recurrence_matrix
    
    # Create subplot
    ax1 = plt.subplot(2, 1, 1)
    librosa.display.specshow(recurrence_matrix_filtered, 
                            x_axis='time', 
                            y_axis='time',
                            cmap='hot', sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
    plt.colorbar(label='Affinity')
    plt.title(f'MFCC Recurrence Matrix (sigma={SIGMA}) - {track_name}')
    plt.xlabel('Time (s)')
    plt.ylabel('Time (s)')

    print("Cuepoints: ", cuepoints)
    print("Changepoints: ", list(changepoints))
    
    # Plot cuepoints
    for cuepoint in cuepoints:
        plt.axvline(x=cuepoint, color='blue', linestyle='--', linewidth=4, alpha=0.5)
        plt.axhline(y=cuepoint, color='blue', linestyle='--', linewidth=4, alpha=0.5)
    
    # Plot changepoints
    for cuepoint in changepoints:
        plt.axvline(x=cuepoint, color='green', linestyle='--', linewidth=2, alpha=1)
        plt.axhline(y=cuepoint, color='green', linestyle='--', linewidth=2, alpha=1)
    
    # Subplot 2: RMS Energy
    ax2 = plt.subplot(2, 1, 2)
    
    # Load audio and compute RMS
    y, sr = librosa.load(filepath, sr=SAMPLE_RATE)
    rms = get_smoothed_rms(filepath)
    times = librosa.frames_to_time(np.arange(len(rms)), sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
    
    # Plot RMS
    ax2.plot(times, rms, linewidth=1.5, color='purple', label='RMS Energy')
    ax2.fill_between(times, rms, alpha=0.3, color='purple')
    
    # Plot cuepoints on RMS
    for i, cuepoint in enumerate(cuepoints):
        ax2.axvline(x=cuepoint, color='blue', linestyle='--', linewidth=4, alpha=0.5, 
                   label='Cuepoint' if i == 0 else '')
    
    # Plot changepoints on RMS
    for i, changepoint in enumerate(changepoints):
        ax2.axvline(x=changepoint, color='green', linestyle='--', linewidth=2, alpha=1, 
                   label='Changepoint' if i == 0 else '')
    
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('RMS Energy')
    ax2.set_title('RMS Energy over Time')
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    print("end visualizing")
    plt.show()

# track_path = "/Users/shivamenta/Desktop/Super Bass.mp3"
# track_name = "MPH – One Sixty.mp3"
# cuepoints = load_rekordbox_training_cuepoints(track_path)
# first_beat_timestamps = load_rekordbox_first_beat_timestamps(track_path)
# recurrence_matrix = get_recurrence_matrix(track_path)
# changepoints = get_changepoints_from_recurrence_matrix(recurrence_matrix, cuepoints, first_beat_timestamps, track_path)
# changepoints = fit_changepoints_to_beatgrid(changepoints, first_beat_timestamps)
# # changepoints = []
# plot_recurrence_matrix(recurrence_matrix, track_name, cuepoints, changepoints, track_path, smooth=False)

track_path = "/Users/shivamenta/Desktop/vocals.wav"
cuepoints = load_rekordbox_training_cuepoints("/Users/shivamenta/Desktop/training_data/Super Bass.mp3")
recurrence_matrix = get_recurrence_matrix(track_path)
changepoints = get_changepoints_from_recurrence_matrix(recurrence_matrix, [], [], track_path)
# changepoints = fit_changepoints_to_beatgrid(changepoints, [])
# changepoints = []
plot_recurrence_matrix(recurrence_matrix, "test", cuepoints, changepoints, track_path, smooth=False)


recurrence_matrix = get_recurrence_matrix("/Users/shivamenta/Desktop/instrumentals.wav")
changepoints = get_changepoints_from_recurrence_matrix(recurrence_matrix, [], [], track_path)
plot_recurrence_matrix(recurrence_matrix, "test", cuepoints, changepoints, track_path, smooth=False)